# Fine-tune `unsloth/gemma-4-E4B-it` with Unsloth

Free Google **Colab T4** notebook. You don't need a GPU at home — Colab lends you one.

**What happens here (plain English):**
1. Install Unsloth (a fast, low-memory fine-tuning library).
2. Load the base model in 4-bit (so it fits a free GPU).
3. Add a **LoRA** adapter — a small trainable add-on, so we teach new behaviour cheaply.
4. Upload your `train.jsonl` (exported from the eval dashboard) and format it.
5. Train for a few minutes.
6. Export a **GGUF** file and download it — that's the file LM Studio loads.

**Before you start:** menu **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**.


### 1. Install Unsloth


In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps --upgrade peft accelerate bitsandbytes
!pip install --force-reinstall --no-deps "trl==0.24.0"   # pin the API this notebook uses


### 2. Load the base model (4-bit, fits a free T4)
If the base model is **gated** on Hugging Face (e.g. Llama), uncomment the login line
and paste your free HF token.


In [ ]:
from unsloth import FastLanguageModel
import torch
# from huggingface_hub import login; login()   # uncomment for gated base models

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)


### 3. Add the LoRA adapter (the small trainable part)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


### 4. Upload your data and format it
Run the cell, then pick the **`companion_train.jsonl`** you exported from the dashboard
(Fine-tune tab → *Export train as JSONL*). Each line is a chat with user + assistant turns.


In [ ]:
from google.colab import files
uploaded = files.upload()   # choose your train.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="companion_train.jsonl", split="train")

def fmt(ex):
    msgs = ex["messages"]
    # Gemma has no 'system' role -> fold a leading system turn into the first user turn.
    if msgs and msgs[0]["role"] == "system":
        sys_txt = msgs[0]["content"]; rest = msgs[1:]
        if rest and rest[0]["role"] == "user":
            rest = [{"role": "user", "content": sys_txt + "\n\n" + rest[0]["content"]}] + rest[1:]
        msgs = rest
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(fmt)
print(dataset[0]["text"][:600])


### 5. Train (~a few minutes for a small set, 2 epoch(s))


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    processing_class = tokenizer,   # trl >=0.13 renamed `tokenizer`
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, num_train_epochs = 2, learning_rate = 0.0002,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01,
        lr_scheduler_type = "linear", seed = 3407, output_dir = "outputs",
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
    ),
)
trainer.train()


### 6. Export to GGUF (`q4_k_m`) and download → load into LM Studio
The download is the model file. In LM Studio: **My Models → Import**, then it appears
in the dashboard's model list as your fine-tuned model for the *after* eval.


In [ ]:
model.save_pretrained_gguf("mira-gemma4-e4b", tokenizer, quantization_method="q4_k_m")
import os
for f in os.listdir("mira-gemma4-e4b"):
    if f.endswith(".gguf"):
        print("downloading", f); files.download(f"mira-gemma4-e4b/" + f)


---
Back in the dashboard **🎓 Fine-tune** tab: pick this model under *Eval (after)*, run it,
and step 4 tells you — with statistics — whether the fine-tune **really** helped.
